## Assignment 1:

- Using Numpy to implement the soft-margin SVM model. 
- Train this model using SGD method on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). Resize the images to $128 \times 128$.
- Evaluate this model using Precision, Recall, and F1 metrics.

### Phương pháp tiền xử lý dữ liệu ảnh: 
- Image Resizing về kích thước 128x128 cho đúng với yêu đầu bài 
- Normaliztion: chuyển từ [0,255] về [0,1] để hội tụ nhanh hơn
- Image flatten: chuyển từ ma trận 2D thành 1D để đưa vào thuật toán SVM : từ (128x128) -> (16348, 0)
- Data shuffling: Hoán đổi vị trí indices để tránh bị overfitting 

In [4]:
import numpy as np
import os
from PIL import Image
from sklearn.preprocessing import StandardScaler

class ChestXRayDataLoader:
    
    def __init__(self, img_size=128, normalize=True):
        self.img_size = img_size
        self.normalize = normalize
        self.scaler = StandardScaler()
        self.X = None
        self.y = None
        
    def load_from_directory(self, data_dir):
        X = []
        y = []
        
        normal_dir = os.path.join(data_dir, 'NORMAL')
        if os.path.exists(normal_dir):
            normal_files = [f for f in os.listdir(normal_dir) 
                          if f.lower().endswith(('.jpeg', '.jpg', '.png'))]
            for img_file in normal_files:
                img_path = os.path.join(normal_dir, img_file)
                try:
                    img = self._load_and_preprocess_image(img_path)
                    X.append(img)
                    y.append(0)
                except Exception as e:
                    pass
        
        pneumonia_dir = os.path.join(data_dir, 'PNEUMONIA')
        if os.path.exists(pneumonia_dir):
            pneumonia_files = [f for f in os.listdir(pneumonia_dir) 
                             if f.lower().endswith(('.jpeg', '.jpg', '.png'))]
            for img_file in pneumonia_files:
                img_path = os.path.join(pneumonia_dir, img_file)
                try:
                    img = self._load_and_preprocess_image(img_path)
                    X.append(img)
                    y.append(1)
                except Exception as e:
                    pass
        
        self.X = np.array(X, dtype=np.float32)
        self.y = np.array(y)

        indices = np.random.permutation(len(self.X))
        self.X = self.X[indices]
        self.y = self.y[indices]
        
        print(f"   NORMAL: {np.sum(self.y == 0)}, PNEUMONIA: {np.sum(self.y == 1)}")
        print(f"   Feature shape: {self.X.shape}")
        
        return self.X, self.y
    
    def _load_and_preprocess_image(self, img_path):
        img = Image.open(img_path).convert('L')
        img = img.resize((self.img_size, self.img_size))
        img_array = np.array(img, dtype=np.float32)
        if self.normalize:
            img_array = img_array / 255.0
        return img_array.flatten()
    
    def scale_data(self, X_train=None):
        if X_train is not None:
            self.scaler.fit(X_train)
            self.X = self.scaler.transform(self.X)
        else:
            self.X = self.scaler.fit_transform(self.X)
        return self.X
    
    def get_train_test_split(self, test_size=0.2, random_state=42):
        from sklearn.model_selection import train_test_split
        
        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y, 
            test_size=test_size,
            random_state=random_state,
            stratify=self.y 
        )
        
        print(f"   Train: {len(X_train)} samples")
        print(f"   Test: {len(X_test)} samples")
        
        return X_train, X_test, y_train, y_test
    
    def get_data(self):
        return self.X, self.y
    
    def shuffle_data(self, random_state=42):
        indices = np.random.RandomState(random_state).permutation(len(self.X))
        self.X = self.X[indices]
        self.y = self.y[indices]
        return self.X, self.y


### Ý tưởng triển khai thuật toán: 
- Em sử dung SGD mini-batch để cập nhật trọng số w và b. Với SGD mini-batch thì trọng số w và b được cập nhật theo mỗi batch. 
- Object Function bao gồm Cost Function(Hinge Loss) + Regularization (l2) để tránh hiện tượng overfitting. 
- Nếu Điều kiện ràng buộc là đúng thì Hinge Loss = 0 và cập nhật dw chỉ còn là lamda nhân w, nếu có bất kỳ điểm dữ liệu nào mini-batch không thỏa mãn ràng buộc thì còn cộng thêm đạo hàm của hinge loss. 

In [5]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
class SVM_SGD_Vectorized:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000, batch_size=32):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.batch_size = batch_size
        self.w = None
        self.b = None

    def train(self, X, y):
        n_samples, n_features = X.shape
        
        y_ = np.where(y <= 0, -1, 1)

        self.w = np.zeros(n_features)
        self.b = 0

        for epoch in range(self.n_iters):
            indices = np.random.permutation(n_samples) 
            X_shuffled = X[indices]
            y_shuffled = y_[indices]
 
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i:i+self.batch_size]
                y_batch = y_shuffled[i:i+self.batch_size]

                margins = y_batch * (X_batch @ self.w + self.b)

                mask = margins < 1

                dw = 2 * self.lambda_param * self.w
                db = 0

            if np.any(mask):
                X_mis = X_batch[mask]
                y_mis = y_batch[mask]

                hinge_grad_w = -np.sum(X_mis * y_mis[:, np.newaxis], axis=0) / self.batch_size
                hinge_grad_b = -np.sum(y_mis) / self.batch_size
                
                dw += hinge_grad_w
                db += hinge_grad_b

            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict(self, X):
        return np.sign(X @ self.w + self.b)

    def evaluate(self, X, y):
        y_ = np.where(y <= 0, -1, 1)
        predictions = self.predict(X)
        
        y_pred = np.where(predictions <= 0, 0, 1)
        
        accuracy = accuracy_score(y, y_pred)
        precision = precision_score(y, y_pred)
        recall = recall_score(y, y_pred)
        f1 = f1_score(y, y_pred)
        
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

train_path = r'E:\HOCMAYTHONGKEREPONOPBAI\BTTH3\archive (1)\chest_xray\train'
test_path = r'E:\HOCMAYTHONGKEREPONOPBAI\BTTH3\archive (1)\chest_xray\test'

loader_train = ChestXRayDataLoader(img_size=128, normalize=True)
X_train, y_train = loader_train.load_from_directory(train_path)

loader_test = ChestXRayDataLoader(img_size=128, normalize=True)
X_test, y_test = loader_test.load_from_directory(test_path)

X_train_scaled = loader_train.scale_data()
X_test_scaled = loader_test.scale_data(X_train=X_train)


   NORMAL: 1341, PNEUMONIA: 3875
   Feature shape: (5216, 16384)
   NORMAL: 234, PNEUMONIA: 390
   Feature shape: (624, 16384)


In [7]:

svm = SVM_SGD_Vectorized(
    learning_rate=0.0001, 
    lambda_param=0.01, 
    n_iters=50,  
    batch_size=32
)
svm.train(X_train_scaled, y_train)

train_metrics = svm.evaluate(X_train_scaled, y_train)
print(f"  Accuracy:  {train_metrics['accuracy']:.4f}")
print(f"  Precision: {train_metrics['precision']:.4f}")
print(f"  Recall:    {train_metrics['recall']:.4f}")
print(f"  F1-Score:  {train_metrics['f1']:.4f}")


test_metrics = svm.evaluate(X_test_scaled, y_test)
print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall:    {test_metrics['recall']:.4f}")
print(f"  F1-Score:  {test_metrics['f1']:.4f}")



  Accuracy:  0.8321
  Precision: 0.9883
  Recall:    0.7832
  F1-Score:  0.8739
  Accuracy:  0.8157
  Precision: 0.9257
  Recall:    0.7667
  F1-Score:  0.8387


## Assigment 2:
- Implement SVM method using a machine learning library (such as sklearn or sktorch).
- Train this model on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). Resize the images to $128 \times 128$.
- Evaluate this model using Precision, Recall, and F1 metrics.
- Compare the results of SVM using library to those of implemented SVM.

In [9]:
from sklearn.svm import SVC

model = SVC(kernel = 'rbf', C = 1.0, gamma= 'scale')
model.fit(X_train_scaled, y_train)


,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [10]:
from sklearn.metrics import classification_report

y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)


print(classification_report(y_train, y_train_pred))

print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      1341
           1       0.99      1.00      0.99      3875

    accuracy                           0.99      5216
   macro avg       0.99      0.99      0.99      5216
weighted avg       0.99      0.99      0.99      5216

              precision    recall  f1-score   support

           0       0.95      0.43      0.59       234
           1       0.74      0.99      0.85       390

    accuracy                           0.78       624
   macro avg       0.85      0.71      0.72       624
weighted avg       0.82      0.78      0.75       624

